In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 72
from IPython.display import display
from collections import defaultdict, Counter
import nltk
from nltk.util import ngrams
import random
import os

## Example I N-gram Text Generation

In [2]:
Corpus1 = [
    "I enjoy learning text_generation.",
    "You enjoy learning GenAI.",
    "Everyone enjoys learning AI."
]

In [3]:
# tokenization
tokens = []
for sentence in Corpus1:
    words = nltk.word_tokenize(sentence.lower())
    tokens.append(['<s>', '<s>'] + words + ['</s>'])

# frequencies
trigram_freq = defaultdict(Counter)
for token_list in tokens:
    for w1, w2, w3 in ngrams(token_list, 3):
        trigram_freq[(w1, w2)][w3] += 1

trigram_prob = defaultdict(dict)
for context, counter in trigram_freq.items():
    total = float(sum(counter.values()))
    for word, count in counter.items():
        trigram_prob[context][word] = count / total

# text generation function
def generate_text1(trigram_prob, max_length=15):
    text = ['<s>', '<s>']
    for _ in range(max_length):
        context = (text[-2], text[-1])
        next_words = list(trigram_prob.get(context, {}).keys())
        if not next_words:
            break
        probabilities = list(trigram_prob[context].values())
        next_word = random.choices(next_words, probabilities)[0]
        if next_word == '</s>':
            break
        text.append(next_word)
    return ' '.join(text[2:])

In [4]:
# run a few times
generated_text = generate_text1(trigram_prob)
print(generated_text) 

you enjoy learning text_generation .


## Example II Naive Bayes Text Generation

In [5]:
Corpus2 = {
    'Positive': [
        'I love this class',
        'This topic is fun'
    ],
    'Negative': [
        'I dislike this class',
        'This topic is hard'
    ]
}

In [6]:
# tokenization
vocabulary = set()
word_counts = defaultdict(lambda: defaultdict(int))
class_word_counts = defaultdict(int)

for cls, sentences in Corpus2.items():
    for sentence in sentences:
        tokens = nltk.word_tokenize(sentence.lower())
        for word in tokens:
            vocabulary.add(word)
            word_counts[cls][word] += 1
            class_word_counts[cls] += 1

vocab_size = len(vocabulary)

# Calculate P(w|C) with Laplace smoothing
word_probs = defaultdict(dict)
for cls in Corpus2.keys():
    total_count = class_word_counts[cls] + vocab_size  # For Laplace smoothing
    for word in vocabulary:
        count = word_counts[cls][word] + 1  # Add-one smoothing
        word_probs[cls][word] = count / total_count

# text generation function
def generate_text2(class_label, length=5):
    generated_words = []
    for _ in range(length):
        words = list(word_probs[class_label].keys())
        probabilities = list(word_probs[class_label].values())
        word = random.choices(words, weights=probabilities, k=1)[0]
        generated_words.append(word)
    return ' '.join(generated_words)

In [7]:
# run a few times
print("Generated Positive Sentence:")
print(generate_text2('Positive'))

print("\nGenerated Negative Sentence:")
print(generate_text2('Negative'))

Generated Positive Sentence:
i fun class hard is

Generated Negative Sentence:
class class dislike dislike hard


## Example III Transformers

In [8]:
import torch
import torch.nn as nn

In [9]:
class TransformerTextGenerator(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_seq_length):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_length, d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)
    
    def forward(self, tgt, memory):
        seq_length = tgt.size(0)
        positions = torch.arange(0, seq_length).unsqueeze(1).to(tgt.device)
        tgt_embedded = self.embedding(tgt) * torch.sqrt(torch.tensor(self.d_model, dtype=torch.float))
        tgt_embedded += self.pos_embedding(positions)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_length).to(tgt.device)
        output = self.transformer_decoder(tgt_embedded, memory, tgt_mask=tgt_mask)
        logits = self.fc_out(output)
        return logits

In [11]:
# sample data
vocab = {'<sos>':0, 'I':1, 'love':2, 'NLP':3, '.':4, '<eos>':5}
inv_vocab = {v: k for k, v in vocab.items()}

In [12]:
# hyperparameters
vocab_size = len(vocab)
d_model = 8
nhead = 2
num_layers = 2
max_seq_length = 10

# initialize the model
model = TransformerTextGenerator(vocab_size, d_model, nhead, num_layers, max_seq_length)

# sample training data
sentences = [
    ['<sos>', 'I', 'love', 'NLP', '.', '<eos>'],
    ['<sos>', 'I', 'love', 'NLP', '<eos>']
]

# tokenize
def tokenize(_sent):
    return [vocab[w] for w in _sent]

tokenized_sentences = [torch.tensor(tokenize(s), dtype=torch.long) for s in sentences]

# loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# train
for epoch in range(100):
    totloss = 0
    for sentence in tokenized_sentences:
        optimizer.zero_grad()
        tgt_input = sentence[:-1].unsqueeze(1)  # exclude last token
        tgt_output = sentence[1:].unsqueeze(1)  # exclude first token

        # since we are only using the decoder, memory can be zeros
        memory = torch.zeros(1, 1, d_model)

        outputs = model(tgt_input, memory)
        loss = criterion(outputs.view(-1, vocab_size), tgt_output.view(-1))
        loss.backward()
        optimizer.step()
        totloss += loss.item()
    if (epoch+1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}, Loss: {totloss:.3f}')

Epoch  20, Loss: 1.716
Epoch  40, Loss: 1.283
Epoch  60, Loss: 1.120
Epoch  80, Loss: 0.878
Epoch 100, Loss: 0.807


In [13]:
# text generation
def generate_text3(model, start_token, max_length=10):
    model.eval()
    generated = [vocab[start_token]]
    memory = torch.zeros(1, 1, d_model)
    for _ in range(max_length):
        tgt_input = torch.tensor(generated).unsqueeze(1)
        outputs = model(tgt_input, memory)
        next_token_logits = outputs[-1, 0, :]
        next_token = torch.argmax(next_token_logits).item()
        generated.append(next_token)
        if next_token == vocab['<eos>']:
            break
    gen_text = [inv_vocab[idx] for idx in generated]
    return ' '.join(gen_text[1:-1])  # Exclude <sos> and <eos>

In [14]:
# run a few times
print("Generated Text:")
print(generate_text3(model, '<sos>'))

Generated Text:
I love NLP



## Example IV Transformers with GPT2

In [4]:
from huggingface_hub import HfApi
api = HfApi()
try:
    who = api.whoami()
    print("Logged in as:", who["name"])
except Exception as e:
    print("Not authenticated:", e)

Logged in as: InfestedCoder


In [3]:
import os
from pathlib import Path

MODEL_PATH= 'EP_models/'
os.environ['HF_HOME']= MODEL_PATH  # before import transformers
os.environ['HF_DATASETS_CACHE']= 'EP_datasets/'  # dataset cache

import transformers
from transformers import GPT2LMHeadModel, GPT2Tokenizer 

# filter warnings
transformers.logging.set_verbosity_error()

print(f'transformers version= {transformers.__version__}')

transformers version= 5.0.0


In [6]:
model_name = 'gpt2'  # or 'gpt2-medium', 'gpt2-large', etc.

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [7]:
# display the model
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [8]:
# generate text
def generate_text4(prompt, max_length=50):
    # Encode the input prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt')
    # generate text
    output = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        early_stopping=False
    )
    # decode the generated text
    gen_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return gen_text

In [9]:
# run a few times
prompt = "I love NLP"
generated_text = generate_text4(prompt)
print("Generated Text:\n")
print(generated_text)    

Generated Text:

I love NLP, but I don't think it's the best way to do it. I think the way it works is that you have to have a lot of people who are willing to listen to you, and you can't just sit back


## Example V LangChain for Text Generation

In [10]:
from langchain_core.language_models.llms import LLM
from langchain_core.prompts import PromptTemplate
from typing import Optional, List, Mapping, Any
import random
from collections import defaultdict, Counter

In [11]:
Corpus5 = [
    "alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, 'and what is the use of a book,' thought alice 'without pictures or conversation?",
    "so she was considering in her own mind (as well as she could, for the hot day made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of getting up and picking the daisies, when suddenly a white rabbit with pink eyes ran close by her.",
    "there was nothing so VERY remarkable in that; nor did Alice think it so VERY much out of the way to hear the Rabbit say to itself, 'Oh dear!",
    "oh dear!",
    "i shall be late!' (when she thought it over afterwards, it occurred to her that she ought to have wondered at this, but at the time it all seemed quite natural); but when the Rabbit actually took a watch out of its waistcoat-pocket, and looked at it, and then hurried on, alice started to her feet, for it flashed across her mind that she had never before seen a rabbit with either a waistcoat-pocket, or a watch to take out of it, and burning with curiosity, she ran across the field after it, and fortunately was just in time to see it pop down a large rabbit-hole under the hedge.",
    "in another moment down went alice after it, never once considering how in the world she was to get out again.",
    "the rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that alice had not a moment to think about stopping herself before she found herself falling down a very deep well."
]

In [12]:
# tokenization and build a trigram model
def build_trigram_model(corpus):
    trigram_counts = defaultdict(Counter)
    for text in corpus:
        tokens = text.lower().split()
        tokens = ['<s>', '<s>'] + tokens + ['</s>']
        for i in range(len(tokens)-2):
            context = (tokens[i], tokens[i+1])
            word = tokens[i+2]
            trigram_counts[context][word] += 1
    # Convert counts to probabilities
    trigram_prob = defaultdict(dict)
    for context, counter in trigram_counts.items():
        total = float(sum(counter.values()))
        for word, count in counter.items():
            trigram_prob[context][word] = count / total
    return trigram_prob

Trigram_model = build_trigram_model(Corpus5)

In [13]:
# A custom LLM
class NGramLLM(LLM):
    trigram_model: dict  # Declare as a class attribute

    @property
    def _llm_type(self) -> str:
        return "n-gram"

    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        return {"name": "NGramLLM"}

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        # Tokenize the prompt
        tokens = prompt.lower().split()
        tokens = ['<s>'] + tokens
        if len(tokens) >= 2:
            context = (tokens[-2], tokens[-1])
        else:
            context = ('<s>', tokens[-1])
        generated_tokens = []
        for _ in range(50):  # Limit to 50 tokens for longer generation
            next_words = self.trigram_model.get(context, {})
            if not next_words:
                break
            words = list(next_words.keys())
            probabilities = list(next_words.values())
            next_word = random.choices(words, weights=probabilities, k=1)[0]
            if next_word == '</s>' or (stop and next_word in stop):
                break
            generated_tokens.append(next_word)
            context = (context[1], next_word)
        return ' '.join(generated_tokens)

# create an instance
llm = NGramLLM(trigram_model=Trigram_model)

In [14]:
# a prompt template
template = """{input_text}"""

prompt = PromptTemplate(
    input_variables=["input_text"],
    template=template,
)

In [15]:
# create an LLM chain
chain = prompt | llm

In [16]:
# generate text
input_prompt = "Alice"

generated_text = chain.invoke(input_prompt)
print("Generated Text:\n")
print(f"{input_prompt} {generated_text}")

transformers version= 5.0.0


In [17]:
# Try different prompts
prompts = ["The rabbit", "She was", "In another", "Suddenly"]

for prompt_text in prompts:
    generated = chain.invoke(prompt_text)
    print(f"\nPrompt: {prompt_text}")
    print(f"Generated Text:\n{prompt_text} {generated}")


Prompt: The rabbit
Generated Text:
The rabbit say to itself, 'oh dear!

Prompt: She was
Generated Text:
She was to get out again.

Prompt: In another
Generated Text:
In another moment down went alice after it, never once considering how in the world she was considering in her own mind (as well as she could, for the hot day made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of getting

Prompt: Suddenly
Generated Text:
Suddenly 
